human lung cancer

导入相关包

In [1]:
from pathlib import Path
import warnings
import os
import scanpy as sc
import scib
import numpy as np
import pandas as pd
import sys
import scgpt as scg
import matplotlib.pyplot as plt
import anndata

import time
import torch
from memory_profiler import memory_usage
from datetime import datetime
from functools import wraps

plt.style.context('default')
warnings.simplefilter("ignore", ResourceWarning)

model_dir = Path("/home/cavin/jt/benchmark/models/SCGPT")


def gene2ensembl(gene_list,mapping_path):
    mapping_df = pd.read_csv(mapping_path,header=0)
    mapping_df_unique = mapping_df.drop_duplicates(subset=['Gene name'], keep='first')
    mapping_dict = dict(zip(mapping_df_unique['Gene name'], mapping_df_unique['ensembl ID']))
    mapped_gene_list = []
    unmapped_genes = []
    for gene in gene_list:
        if gene.startswith('ENS'):
            mapped_gene_list.append(gene)
            continue
        if gene in mapping_dict:
            mapped_gene_list.append(mapping_dict[gene])
        else:
            mapped_gene_list.append(gene)
            unmapped_genes.append(gene)
    if unmapped_genes:
        print(f"Warning: {len(unmapped_genes)} genes not found in mapping file: {unmapped_genes[:10]}{'...' if len(unmapped_genes) > 10 else ''}")
    return mapped_gene_list
    


def measure_resources(cuda_id: int = 0,task_des:str="task",save_dir:str="/home/cavin/jt/benchmark/experiments/results/run_metric/cluster_with_annotation"):
    """
    测量函数运行时的CPU内存、指定GPU的显存消耗(峰值)和运行时间。
    
    参数:
    cuda_id (int): 需要监控的 GPU 编号，例如传入 0 代表监控 cuda:0
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            
            
            device = None
            if torch.cuda.is_available():
                try:
                    if not torch.cuda.is_initialized():
                        torch.cuda.init()
                    device = torch.device(f"cuda:{cuda_id}")
                    torch.cuda.reset_peak_memory_stats(device)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 初始化监控失败: {e}")

            start_time = time.time()
            
            mem_usage_mb, result = memory_usage(
                (func, args, kwargs), 
                max_usage=True, 
                retval=True
            )
            
            end_time = time.time()
            execution_time = end_time - start_time
            
           
            mem_usage_gb = mem_usage_mb / 1024.0
            
            allocated_gb = 0.0
            cached_gb = 0.0

            
            if device is not None:
                try:
                    allocated_gb = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
                    cached_gb = torch.cuda.max_memory_reserved(device) / (1024 ** 3)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 显存统计失败: {e}")

            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # ===========================
            # 美观的表格式输出
            # ===========================
            report = f"""
============================================================
RESOURCE USAGE REPORT: {func.__name__}
============================================================
Timestamp                   : {timestamp}
Target GPU ID               : {cuda_id}
Execution Time (Minutes)    : {execution_time/60:.4f}
Execution Time (Seconds)    : {execution_time:.2f}
CPU Peak Memory Usage (GB)  : {mem_usage_gb:.4f}
GPU Peak Allocated Mem (GB) : {allocated_gb:.4f}
GPU Peak Cached Mem (GB)    : {cached_gb:.4f}
CUDA Available              : {torch.cuda.is_available()}
============================================================
"""
            print(report)
            save_file_name = "record.csv"
            full_path = os.path.join(save_dir, save_file_name)
            os.makedirs(save_dir, exist_ok=True)
            new_record = {"Timestamp":timestamp,"Task":task_des,
                          "Target GPU ID ":cuda_id,
                          "Execution Time (Minutes)":execution_time / 60,
                          "Execution Time (Seconds)":execution_time,
                          "CPU Peak Memory Usage (GB)":mem_usage_gb,
                          "GPU Peak Allocated Mem (GB)":allocated_gb,
                          "GPU Peak Cached Mem (GB)":cached_gb}
            new_data_df = pd.DataFrame([new_record])
            if not os.path.isfile(full_path):
                # 如果文件不存在：直接正常写入（默认 mode='w'），自带表头
                new_data_df.to_csv(full_path, index=False)
                print(f"文件不存在，已创建新文件并写入表头：{full_path}")
            else:
                # 如果文件已存在：使用追加模式 (mode='a')，并且绝对不写表头 (header=False)
                new_data_df.to_csv(full_path, mode='a', header=False, index=False)
                print(f"文件已存在，数据已成功追加至末尾。")
            return result
        return wrapper
    return decorator

读取数据和数据预处理

In [2]:
simple_path = '/home/cavin/jt/benchmark/data/human_lung_cancer/SMI_Lung.h5ad'
adata = sc.read_h5ad(simple_path)
gene_col = "gene_name"
cell_type_key = "cell_type"
# batch_key = "sample"
N_HVG = 2000
celltype_id_labels = adata.obs[cell_type_key].astype("category").cat.codes.values
adata = adata[celltype_id_labels >= 0]
org_adata = adata.copy()
adata

View of AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    obsm: 'spatial'

In [3]:
# highly variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor='seurat_v3')
adata = adata[:, adata.var['highly_variable']]

/home/cavin/anaconda3/envs/scgpt/lib/python3.10/site-packages/scanpy/preprocessing/_highly_variable_genes.py:178: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["hvg"] = {"flavor": flavor}


In [4]:
adata.var_names

Index(['AATK', 'ABL1', 'ABL2', 'ACE', 'ACE2', 'ACKR1', 'ACKR3', 'ACKR4',
       'ACTA2', 'ACTG2',
       ...
       'WNT5B', 'WNT7A', 'WNT7B', 'WNT9A', 'XBP1', 'XCL1', 'XCL2', 'YBX3',
       'YES1', 'ZFP36'],
      dtype='object', length=960)

In [5]:
adata_idlist = gene2ensembl(adata.var_names.to_list(),"/home/cavin/jt/benchmark/data/new_genes_homo_sapiens.csv")

In [6]:
adata.var['ensembl_id'] = pd.DataFrame(adata_idlist, index=adata.var.index)
adata.var[gene_col] = adata.var.index
adata.var.index = adata_idlist
adata.var_names.name = None
adata.var_names

/tmp/ipykernel_2616132/1951966021.py:1: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var['ensembl_id'] = pd.DataFrame(adata_idlist, index=adata.var.index)


Index(['ENSG00000181409', 'ENSG00000097007', 'ENSG00000143322',
       'ENSG00000159640', 'ENSG00000130234', 'ENSG00000213088',
       'ENSG00000144476', 'ENSG00000129048', 'ENSG00000107796',
       'ENSG00000163017',
       ...
       'ENSG00000111186', 'ENSG00000154764', 'ENSG00000188064',
       'ENSG00000143816', 'ENSG00000100219', 'ENSG00000143184',
       'ENSG00000143185', 'ENSG00000060138', 'ENSG00000176105',
       'ENSG00000128016'],
      dtype='object', length=960)

提取嵌入

In [7]:
@measure_resources(task_des="SCGpt run cosmx human lung cancer")
def get_emb():
    embed_adata = scg.tasks.embed_data(
        adata,
        model_dir,
        gene_col=gene_col,
        batch_size=16,
    )
    return embed_adata.obsm["X_scGPT"]

In [8]:
emb = get_emb()

scGPT - INFO - match 958/960 genes in vocabulary of size 60697.


Embedding cells: 100%|██████████| 5475/5475 [00:56<00:00, 96.24it/s] 
/home/cavin/anaconda3/envs/scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings



RESOURCE USAGE REPORT: get_emb
Timestamp                   : 2026-03-08 18:57:00
Target GPU ID               : 0
Execution Time (Minutes)    : 1.0371
Execution Time (Seconds)    : 62.22
CPU Peak Memory Usage (GB)  : 5.2357
GPU Peak Allocated Mem (GB) : 0.3615
GPU Peak Cached Mem (GB)    : 0.4883
CUDA Available              : True

文件已存在，数据已成功追加至末尾。


保存嵌入

In [9]:
emb.shape

(87589, 512)

In [10]:
emb_df = pd.DataFrame(
    emb, 
    index=adata.obs_names,
    columns=[f"scGPT_dim_{i}" for i in range(emb.shape[1])]
)
print(emb_df.head())
save_path = "/home/cavin/jt/benchmark/experiments/embedings/spatial_cluster_with_annotations/scgpt_human_lung_cancer_emb.parquet"
emb_df.to_parquet(save_path, compression="zstd")

           scGPT_dim_0  scGPT_dim_1  scGPT_dim_2  scGPT_dim_3  scGPT_dim_4  \
unique_ID                                                                    
1-2           0.007512     0.043102     0.002787    -0.036312     0.005574   
1-3           0.021533     0.030449     0.002276    -0.045953    -0.001659   
1-4           0.016473     0.023216     0.025212    -0.048167     0.003165   
1-6           0.033939     0.031950    -0.002069    -0.032953     0.018575   
1-7           0.021888     0.042498    -0.005114    -0.038843     0.025823   

           scGPT_dim_5  scGPT_dim_6  scGPT_dim_7  scGPT_dim_8  scGPT_dim_9  \
unique_ID                                                                    
1-2           0.023970    -0.008778     0.008075     0.012466    -0.021002   
1-3           0.026596    -0.013778     0.002263    -0.005564    -0.000645   
1-4          -0.002769    -0.000194    -0.001114    -0.015644    -0.011956   
1-6           0.010902    -0.007614    -0.007632    -0.025255  

读取嵌入

In [11]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    obsm: 'spatial'

In [12]:
adata.obsm["X_scGPT"] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    obsm: 'spatial', 'X_scGPT'

In [13]:
adata.obsm["X_scGPT"]

array([[ 0.00751195,  0.04310248,  0.00278669, ..., -0.02574796,
        -0.0001176 ,  0.02698212],
       [ 0.02153301,  0.03044883,  0.00227639, ..., -0.02093222,
         0.0016582 ,  0.01843485],
       [ 0.016473  ,  0.02321622,  0.02521247, ..., -0.01795154,
         0.03027099,  0.02280515],
       ...,
       [ 0.02339709,  0.03921311, -0.01653948, ..., -0.02727631,
         0.01098404,  0.00556158],
       [ 0.02845841,  0.03374735, -0.00751741, ..., -0.02793336,
         0.04130304,  0.00139459],
       [ 0.01303388,  0.05379374, -0.00797579, ..., -0.03239902,
         0.02333021,  0.01141812]], dtype=float32)